# PolyAtlas — Chen 2024 external validation with ANARCI (v5.0)

Reproduces the Chen 2024 generalization experiment for the PEDS revision, using **ANARCI** (via `abnumber`) for canonical IMGT CDR boundaries.

**This version uses pip, not conda — no kernel restarts.** Just Runtime → Run all, top to bottom. A regular **CPU** runtime is correct. Total ~10–15 min (mostly ANARCI annotation time).

Outputs: AUROC/AUPRC for **Mode A** (frozen NbBench Model 1 applied to Chen, zero refit), **Mode B** (13-feature family refit on Chen), **Mode C** (full 52-feature catalog refit), plus coefficient sign agreement vs NbBench and top-k enrichment.

**Changelog**
- **v5.0** — Dropped condacolab entirely (root cause of every prior failure: it splits the Python environment and clashes with Colab's preinstalled Biopython). v5 installs ANARCI the pure-pip way: `apt install hmmer` for the one system dependency, then `pip install anarci abnumber` into Colab's own Python. No conda, no kernel restart, no sys.path surgery. If the pip ANARCI wheel is unavailable, falls back to installing ANARCI from its GitHub source.
- **v4.0** — conda Biopython pin (still failed: condacolab Python-version split).
- **v3.0** — kernel restart after conda install (still failed: Bio.Align clash).
- **v2.0 / v1.0** — earlier condacolab attempts.

## 1. Install ANARCI via pip (no conda)
`hmmer` is the one system dependency (apt). Then `anarci` + `abnumber` install into Colab's own Python. No restart needed.

In [ ]:
# Cell 1 — pip-based ANARCI install (no conda, no restart)
import sys, subprocess

# 1) HMMER: the one non-python dependency ANARCI needs
!apt-get -qq install -y hmmer > /dev/null 2>&1
print("hmmer:", subprocess.run(["which","hmmscan"],capture_output=True,text=True).stdout.strip() or "NOT FOUND")

# 2) ANARCI + abnumber from pip. The modern PyPI wheel (jnooree fork) is pure-python.
!pip install -q anarci abnumber

# 3) verify import; if the wheel didn't provide the germline HMMs, fall back to GitHub source
try:
    from abnumber import Chain
    print("abnumber import OK (pip)")
except Exception as e:
    print("pip import failed, installing ANARCI from GitHub source:", e)
    !pip install -q biopython
    !git clone -q https://github.com/oxpig/ANARCI.git /content/ANARCI_src
    !cd /content/ANARCI_src && python setup.py install -q
    !pip install -q abnumber
    from abnumber import Chain
    print("abnumber import OK (GitHub source)")

In [ ]:
# Cell 1b — sanity check on trastuzumab VH
from abnumber import Chain
tras = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
c = Chain(tras, scheme="imgt")
print("CDR1:", c.cdr1_seq)
print("CDR2:", c.cdr2_seq)
print("CDR3:", c.cdr3_seq)

## 2. Download the data and the PolyAtlas model coefficients

In [ ]:
import urllib.request, os

CHEN_URL = "https://github.com/Tessier-Lab-UMich/Human_Ab_Polyreactivity/archive/refs/tags/v1.2.0-alpha.zip"
urllib.request.urlretrieve(CHEN_URL, "chen.zip")
!unzip -q -o chen.zip
CHEN_DIR = "Human_Ab_Polyreactivity-1.2.0-alpha"
assert os.path.isdir(CHEN_DIR), "Chen download/unzip failed"
print("Chen files:", os.listdir(CHEN_DIR)[:6])

COEF_URL = "https://raw.githubusercontent.com/avnish-deobhakta/polyatlas-peds/main/models/model1_coefficients.csv"
urllib.request.urlretrieve(COEF_URL, "model1_coefficients.csv")
print("coefficients downloaded")

## 3. Feature-computation code (verbatim from the PolyAtlas repo)

In [ ]:
import numpy as np, pandas as pd

KYTE_DOOLITTLE = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE_AT_PH74 = {"D":-1,"E":-1,"K":1,"R":1,"H":0.1}
AROMATIC=set("FWY"); POSITIVE=set("KR"); NEGATIVE=set("DE"); HYDROPHOBIC=set("ILVFMWYC")
PKA={"C_term":3.55,"D":4.05,"E":4.45,"H":5.98,"K":10.0,"R":12.0,"Y":10.0,"C":9.0,"N_term":8.0}

def net_charge(s):
    if not isinstance(s,str) or not s: return np.nan
    return sum(CHARGE_AT_PH74.get(a,0) for a in s.upper())

def frac(s,sub):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper()
    return sum(1 for a in s if a in sub)/len(s)

def frac_res(s,r):
    if not isinstance(s,str) or not s: return np.nan
    return s.upper().count(r)/len(s)

def mean_hphob(s):
    if not isinstance(s,str) or not s: return np.nan
    return np.mean([KYTE_DOOLITTLE.get(a,0) for a in s.upper()])

def max_hphob_run(s):
    if not isinstance(s,str) or not s: return 0
    best=cur=0
    for a in s.upper():
        if a in HYDROPHOBIC:
            cur+=1; best=max(best,cur)
        else:
            cur=0
    return best

def charge_dipole(s):
    if not isinstance(s,str) or len(s)<4: return 0
    m=len(s)//2
    return net_charge(s[:m])-net_charge(s[m:])

def estimate_pI(s):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper()
    def c_at(ph):
        c=1/(1+10**(ph-PKA["N_term"]))-1/(1+10**(PKA["C_term"]-ph))
        for a in s:
            if a in ("K","R"): c+=1/(1+10**(ph-PKA[a]))
            elif a in ("D","E"): c-=1/(1+10**(PKA[a]-ph))
            elif a=="H": c+=1/(1+10**(ph-PKA["H"]))
            elif a=="Y": c-=1/(1+10**(PKA["Y"]-ph))
            elif a=="C": c-=1/(1+10**(PKA["C"]-ph))
        return c
    lo,hi=0.0,14.0
    for _ in range(50):
        m=(lo+hi)/2
        if c_at(m)>0: lo=m
        else: hi=m
    return (lo+hi)/2

def build_features(df):
    f=pd.DataFrame(index=df.index)
    for region,col in [("H1","CDR1_nogaps"),("H2","CDR2_nogaps"),("H3","CDR3_nogaps"),("full","seq")]:
        s=df[col].fillna("").astype(str)
        f[f"{region}_len"]=s.str.len()
        f[f"{region}_charge"]=s.apply(net_charge)
        f[f"{region}_abs_charge"]=f[f"{region}_charge"].abs()
        f[f"{region}_pos_frac"]=s.apply(lambda x:frac(x,POSITIVE))
        f[f"{region}_neg_frac"]=s.apply(lambda x:frac(x,NEGATIVE))
        f[f"{region}_hphob"]=s.apply(mean_hphob)
        f[f"{region}_hphob_frac"]=s.apply(lambda x:frac(x,HYDROPHOBIC))
        f[f"{region}_arom"]=s.apply(lambda x:frac(x,AROMATIC))
        f[f"{region}_W"]=s.apply(lambda x:frac_res(x,"W"))
        f[f"{region}_R"]=s.apply(lambda x:frac_res(x,"R"))
        f[f"{region}_V"]=s.apply(lambda x:frac_res(x,"V"))
        f[f"{region}_G"]=s.apply(lambda x:frac_res(x,"G"))
    f["H3_charge_dipole"]=df["CDR3_nogaps"].fillna("").astype(str).apply(charge_dipole)
    f["H3_max_hphob_run"]=df["CDR3_nogaps"].fillna("").astype(str).apply(max_hphob_run)
    f["H3_pI"]=df["CDR3_nogaps"].fillna("").astype(str).apply(estimate_pI)
    f["full_pI"]=df["seq"].fillna("").astype(str).apply(estimate_pI)
    return f.fillna(0)

print("feature code ready")

## 4. Load Chen library, subsample, and annotate CDRs with ANARCI
Balanced subsample (40k/40k). Raise `N_PER_CLASS` for the final run; ANARCI runs ~50–100 seq/s on CPU.

In [ ]:
from abnumber import Chain
import time

f = f"{CHEN_DIR}/Supplemental Datasets/Human Ab Poly Dataset S1_v2.xlsx"
df = pd.read_excel(f, sheet_name="Sheet1", header=2)
df["label"] = df["Name"].astype(str).str.contains("high", case=False).astype(int)

N_PER_CLASS = 40000  # bump up (toward ~123000) for the final run
pos = df[df.label==1].sample(n=min(N_PER_CLASS,(df.label==1).sum()), random_state=42)
neg = df[df.label==0].sample(n=min(N_PER_CLASS,(df.label==0).sum()), random_state=42)
sub = pd.concat([pos,neg]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Subsampled to {len(sub)}")

def anarci_cdrs(vh):
    try:
        c = Chain(str(vh), scheme="imgt")
        return c.cdr1_seq, c.cdr2_seq, c.cdr3_seq
    except Exception:
        return None, None, None

t=time.time(); h1=[]; h2=[]; h3=[]
for i,vh in enumerate(sub["VH"].astype(str)):
    a,b,cc = anarci_cdrs(vh)
    h1.append(a); h2.append(b); h3.append(cc)
    if i%5000==0 and i>0:
        print(f"  {i}/{len(sub)}  ({time.time()-t:.0f}s)")
sub["CDR1_nogaps"]=h1; sub["CDR2_nogaps"]=h2; sub["CDR3_nogaps"]=h3
sub["seq"]=sub["VH"].astype(str)
n0=len(sub)
sub=sub[sub.CDR1_nogaps.notna()&sub.CDR2_nogaps.notna()&sub.CDR3_nogaps.notna()].reset_index(drop=True)
print(f"ANARCI annotated {len(sub)}/{n0} ({100*len(sub)/n0:.1f}%) in {time.time()-t:.0f}s")
print("label balance:", sub.label.value_counts().to_dict())
sub.to_csv("chen_annotated.csv", index=False)
print("saved chen_annotated.csv")

## 5. Compute features and run the three evaluation modes

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = build_features(sub); y = sub["label"].values
coef = pd.read_csv("model1_coefficients.csv")
intercept = float(coef[coef.feature=="_intercept"].standardized_coefficient.iloc[0])
coef = coef[coef.feature!="_intercept"].copy()
m1 = list(coef.feature)

# MODE A: frozen NbBench Model 1, zero refit
score = np.full(len(X), intercept, float)
for _,cr in coef.iterrows():
    score += cr.standardized_coefficient*((X[cr.feature].values-cr.train_mean)/cr.train_std)
aA=roc_auc_score(y,score); pA=average_precision_score(y,score)
rng=np.random.RandomState(42); bs=[]
for _ in range(500):
    idx=rng.choice(len(y),len(y),replace=True)
    try: bs.append(roc_auc_score(y[idx],score[idx]))
    except: pass
ciA=(np.percentile(bs,2.5),np.percentile(bs,97.5))

# MODE B: refit the 13-feature family on Chen
Xtr,Xte,ytr,yte=train_test_split(X[m1].values,y,test_size=0.2,random_state=42,stratify=y)
sc=StandardScaler().fit(Xtr)
lr=LogisticRegression(class_weight="balanced",random_state=42,max_iter=500).fit(sc.transform(Xtr),ytr)
sB=lr.predict_proba(sc.transform(Xte))[:,1]; aB=roc_auc_score(yte,sB); pB=average_precision_score(yte,sB)
order=np.argsort(-sB); k5=int(0.05*len(yte)); k10=int(0.10*len(yte))
enr5=yte[order[:k5]].mean()/yte.mean(); enr10=yte[order[:k10]].mean()/yte.mean()

# MODE C: refit full 52-feature catalog (upper bound)
Xtr2,Xte2,ytr2,yte2=train_test_split(X.values,y,test_size=0.2,random_state=42,stratify=y)
sc2=StandardScaler().fit(Xtr2)
lr2=LogisticRegression(class_weight="balanced",random_state=42,max_iter=500).fit(sc2.transform(Xtr2),ytr2)
sC=lr2.predict_proba(sc2.transform(Xte2))[:,1]; aC=roc_auc_score(yte2,sC); pC=average_precision_score(yte2,sC)

achg=roc_auc_score(y,X["H3_charge"]); afull=roc_auc_score(y,X["full_charge"]); api=roc_auc_score(y,X["full_pI"])
print("="*64)
print(f"CHEN 2024 (ANARCI CDRs, n={len(sub)})")
print("="*64)
print(f"MODE A frozen NbBench Model 1:  AUROC={aA:.4f} [{ciA[0]:.4f},{ciA[1]:.4f}]  AUPRC={pA:.4f}")
print(f"MODE B refit 13-feat family:    AUROC={aB:.4f}  AUPRC={pB:.4f}  (top5%={enr5:.2f}x, top10%={enr10:.2f}x)")
print(f"MODE C refit full 52 catalog:   AUROC={aC:.4f}  AUPRC={pC:.4f}")
print(f"zero-train: H3charge={achg:.4f}  full_charge={afull:.4f}  full_pI={api:.4f}  (pos rate={y.mean():.3f})")
print()
print("Coefficient sign agreement (refit-13 vs NbBench):")
rf=dict(zip(m1,lr.coef_[0]))
for feat in m1:
    nb=float(coef[coef.feature==feat].standardized_coefficient.iloc[0])
    tag = "OK" if np.sign(nb)==np.sign(rf[feat]) else "FLIP"
    print(f"  {feat:<18} NbBench={nb:+.3f}  Chen={rf[feat]:+.3f}  {tag}")

import json
json.dump({"version":"v5.0","n":int(len(sub)),
  "modeA_frozen":{"auroc":aA,"auroc_ci":list(ciA),"auprc":pA},
  "modeB_refit13":{"auroc":aB,"auprc":pB,"enr5":enr5,"enr10":enr10},
  "modeC_refit52":{"auroc":aC,"auprc":pC},
  "zero":{"h3_charge":achg,"full_charge":afull,"full_pi":api},
  "pos_rate":float(y.mean()),
  "refit_coef":{k:float(v) for k,v in rf.items()}}, open("chen_results.json","w"), indent=2)
print("\nsaved chen_results.json")

## Notes for the manuscript
- **Mode A** (frozen transfer) is the headline: a model fit only on camelid nanobodies, applied with zero retraining to human antibodies, still separates polyreactive from non-polyreactive.
- **Mode B** shows the same 13-feature family, lightly recalibrated, is stronger still.
- Report Mode A and B as the honest results; Mode C (full 52 refit) is an upper bound.
- For final numbers, raise `N_PER_CLASS` (or remove subsampling) to use the full ~246k library.
- Artifacts: `chen_annotated.csv`, `chen_results.json` — download both for the writeup.